In [ ]:
import sys

sys.path.append("../")

import sympy as sp
import pathlib as pl
from SymEigen import *
from sympy import symbols
from project_dir import backend_source_dir
from affine_body_core import compute_J_point, compute_J_vec

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")
Gen.DisableLatexComment()


ref: [Affine Body Revolute Joint](https://spirimirror.github.io/libuipc-doc/specification/constitutions/affine_body_revolute_joint/)

![Affine Body Revolute Joint](../../docs/specification/constitutions/media/affine_body_revolute_joint_fig1.drawio.svg)

$$
C_0 = (\mathbf{q}_i^0 - \mathbf{q}_j^0 ) = \mathbf{0}
$$

$$
C_1 = (\mathbf{q}_i^1  - \mathbf{q}_j^1) = \mathbf{0}
$$

In [ ]:
kappa = Eigen.Scalar("kappa")

# Basis vectors
qi0_bar = Eigen.Vector("qi0_bar", 3)
qi1_bar = Eigen.Vector("qi1_bar", 3)
qj0_bar = Eigen.Vector("qj0_bar", 3)
qj1_bar = Eigen.Vector("qj1_bar", 3)

# Affine Body DOF vectors
qi = Eigen.Vector("qi", 12)
qj = Eigen.Vector("qj", 12)

$$
C_0 = (\mathbf{q}_i^0 - \mathbf{q}_j^0 ) = \mathbf{0}
$$

$$
C_1 = (\mathbf{q}_i^1  - \mathbf{q}_j^1) = \mathbf{0}
$$
$$
\mathbf{F}_{axis} = \begin{bmatrix}
\mathbf{q}_i^0 - \mathbf{q}_j^0 \\
\mathbf{q}_i^1 - \mathbf{q}_j^1 \\
\end{bmatrix}_{6 \times 1}
$$

Frame Affine Body Mapping:

$$
\mathbf{J}_{axis} = \begin{bmatrix}
\mathbf{J}(\bar{\mathbf{q}}_i^0) & -\mathbf{J}(\bar{\mathbf{q}}_j^0) \\
\mathbf{J}(\bar{\mathbf{q}}_i^1) & -\mathbf{J}(\bar{\mathbf{q}}_j^1) \\
\end{bmatrix}_{6 \times 24}
$$

$$
\mathbf{F}_{axis} = \mathbf{J}_{axis} \cdot
\begin{bmatrix}
\mathbf{q}_i \\
\mathbf{q}_j \\
\end{bmatrix}
$$

In [ ]:
# Mapping
Jaxis = sp.Matrix.zeros(6, 24)
Jaxis[0:3, 0:12] = compute_J_point(qi0_bar)
Jaxis[0:3, 12:24] = -compute_J_point(qj0_bar)
Jaxis[3:6, 0:12] = compute_J_point(qi1_bar)
Jaxis[3:6, 12:24] = -compute_J_point(qj1_bar)


content = ""

# from ABD q to F
Faxis = Jaxis @ sp.Matrix.vstack(qi, qj)
Cl_Faxis = Gen.Closure(
    qi0_bar,
    qi1_bar,
    qi,  # Affine Body i
    qj0_bar,
    qj1_bar,
    qj,
)  # Affine Body j

# from F Gradient to ABD Gradient
Gaxis = Eigen.Vector("Gaxis", 6)
JaxisT_Gaxis = Jaxis.T @ Gaxis
Cl_Gaxis = Gen.Closure(
    Gaxis,
    qi0_bar,
    qi1_bar,  # Affine Body i
    qj0_bar,
    qj1_bar,
)  # Affine Body j
# from F Hessian to ABD Hessian
Haxis = Eigen.Matrix("Haxis", 6, 6)
JaxisT_Haxis_Jaxis = Jaxis.T @ Haxis @ Jaxis
Cl_Haxis = Gen.Closure(
    Haxis,
    qi0_bar,
    qi1_bar,  # Affine Body i
    qj0_bar,
    qj1_bar,
)  # Affine Body j

content += f""" 
// Revolute Joint: C0 C1
// Mapping between ABD qi qj to Faxis

{Cl_Faxis("Faxis", Faxis)}
{Cl_Gaxis("JaxisT_Gaxis", JaxisT_Gaxis)}
{Cl_Haxis("JaxisT_Haxis_Jaxis", JaxisT_Haxis_Jaxis)}

"""

Faxis = Eigen.Vector("Faxis", 6)
dij0 = Faxis[0:3, 0]
dij1 = Faxis[3:6, 0]

Eaxis = 0.5 * kappa * (dij0.dot(dij0) + dij1.dot(dij1))
# E01 = 0
Cl_Eaxis = Gen.Closure(kappa, Faxis)


dEaxisdFaxis = VecDiff(Eaxis, Faxis)
ddEaxisddFaxis = VecDiff(dEaxisdFaxis, Faxis)

content += f""" // Revolute Joint Energy and Derivatives

{Cl_Eaxis("Eaxis", Eaxis)}
{Cl_Eaxis("dEaxisdFaxis", dEaxisdFaxis)}
{Cl_Eaxis("ddEaxisddFaxis", ddEaxisddFaxis)}

// -------------------------------------------------------------------------
"""

In [ ]:
path = backend_source_dir("cuda") / "affine_body/constitutions/sym" / "affine_body_revolute_joint.inl"
with open(path, "w") as f:
    f.write(content)
print(f"Written to {path}")
